# Forecast Using the TiRex Family



foundationforecast ships a single `TiRex` class that transparently supports both TiRex 1.0 and TiRex 2.0 foundation models from NX-AI. Pass the Hugging Face `repo_id` to select the checkpoint:

- **TiRex 1.0** (`NX-AI/TiRex`): univariate zero-shot forecasting with xLSTM.
- **TiRex 2.0** (`NX-AI/TiRex-2`): extends TiRex to multivariate forecasting and covariates (this notebook uses the univariate API).

*Requirements*:

    - Python >= 3.11
    - TiRex 1.0 runs on CPU and CUDA (including macOS).
    - TiRex 2.0 runs on CPU, CUDA, and Apple MPS. On CUDA it uses Triton kernels for speed; on CPU and MPS it uses native PyTorch kernels (no Triton required).

In this example we compare TiRex 1.0 and TiRex 2.0 on event pageview data.



## Import libraries



In [ ]:
import sys

import pandas as pd

from foundationforecast import FoundationForecast

if sys.version_info < (3, 11):
    raise RuntimeError("TiRex requires Python >= 3.11")


## Load the dataset

The DataFrame must include at least the following columns:
- unique_id: Unique identifier for each time series (string)
- ds: Date column (datetime format)
- y: Target variable for forecasting (float format)

The pandas frequency will be inferred from the ds column, if not provided.
If the seasonality is not provided, it will be inferred based on the frequency.
If the horizon is not set, it will default to 2 times the inferred seasonality.



In [ ]:
df = pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()


## Plot the data



In [ ]:
FoundationForecast.plot(df)


## Import the models



In [ ]:
from foundationforecast.models.tirex import TiRex



## Create a FoundationForecast

We compare TiRex 1.0 (`NX-AI/TiRex`) with TiRex 2.0 (`NX-AI/TiRex-2`) Each model gets a distinct `alias` so its forecasts are easy to identify.



In [ ]:
models = [
    TiRex(repo_id="NX-AI/TiRex", alias="NX-AI/TiRex"),
    TiRex(repo_id="NX-AI/TiRex-2", alias="NX-AI/TiRex-2"),]

ff = FoundationForecast(models=models)


## Generate forecast

You can optionally specify the following parameters:
- freq: The frequency of your data (e.g., 'D' for daily, 'M' for monthly)
- h: The forecast horizon, which is the number of periods to predict
- seasonality: The seasonal period of your data, which can be inferred if not provided



In [ ]:
level = [0, 20, 40, 60, 80] # we need to pass 0 level since specific quantiles are required by TiRex
cv_df = ff.cross_validation(df=df, h=12, level=level)


In [ ]:
ff.plot(df, cv_df.drop(columns=["cutoff", "y"]), level=[80])


In [ ]:
cv_df.head()


## Evaluation



In [ ]:
from functools import partial

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, scaled_crps


In [ ]:
eval_df = evaluate(
    cv_df.drop(columns=["cutoff"]),
    train_df=df.query("ds <= '2024-08-31'"),
    metrics=[partial(mase, seasonality=12), scaled_crps],
    level=level,
)
eval_df.groupby("metric").mean(numeric_only=True).T.sort_values(
    by="scaled_crps"
).round(3)
